# Preprocessing the CERT Dataset for Model Training:

### Imports:

In [125]:
import os
import numpy as np
import pandas as pd
from urllib.parse import urlparse

## **Layer A: User-PC-Day Level Dataset**

### Defining Our Constants:

In [126]:
DEFAULT_OUTPUT_DIR = "processed_datasets"
WORK_HOURS = (9, 17)

# Features to use specific to each log file
USECOLS_MAP = {
    "logon":  ["date", "user", "pc", "activity"],
    "file":   ["date", "user", "pc", "activity", "filename"],
    "device": ["date", "user", "pc", "activity"],
    "email":  ["date", "user", "pc", "to", "attachments"],
    "http":   ["date", "user", "pc", "url", "activity"],
}

# Pre-defining data types to reduce memory overhead
DTYPE_MAP = {
    "user":     "category",
    "pc":       "category",
    "activity": "category",
}

# Explicit timestamp format for the CERT dataset
TIMESTAMP_FORMAT = "%m/%d/%Y %H:%M:%S"

# Identifying sources too large to load eagerly
LARGE_FILE_SOURCES = {"email", "http"}

# HTTP specifications
INTERNAL_EMAIL_DOMAIN = "dtaa.com"
LONG_URL_THRESHOLD = 90

JOB_DOMAINS = {
    "careerbuilder.com",
    "indeed.com",
    "monster.com",
    "simplyhired.com",
    "linkedin.com",
    "www.linkedin.com"
}

CLOUD_STORAGE_DOMAINS = {
    "dropbox.com",
    "www.dropbox.com",
    "drive.google.com",
    "docs.google.com",
    "yousendit.com",
    "www.yousendit.com"
}

SUSPICIOUS_DOMAINS = {
    "wikileaks.org",
    "www.wikileaks.org"
}

### Importing the CERT Dataset:

In [127]:
cert_path = r"C:\Users\loera\Documents\Datasets\CERT_r6.2"

In [128]:
def load_raw_logs(cert_path: str) -> dict:
    """
    Loads the raw CERT log files needed for preprocessing. Small files are loaded eagerly
    whereas large files are represented as {path: chunked=True} for downstream chunked
    processing.
    
    Args:
        cert_path: The base path containing the CERT dataset
        
    Returns:
        dict: {file_name: DataFrame | {"path": str, "chunked": True}}
    """
    file_map = {
        "logon":  "logon.csv",
        "file":   "file.csv",
        "device": "device.csv",
        "email":  "email.csv",
        "http":   "http.csv",
    }
    
    logs = {}
    missing_files = []
    
    for source_name, filename in file_map.items():
        full_path = os.path.join(cert_path, filename)
        # Takes note of missing file paths
        if not os.path.exists(full_path):
            missing_files.append(filename)
            continue
        # Defers large files for later chunked loading
        if source_name in LARGE_FILE_SOURCES:
            logs[source_name] = {"path": full_path, "chunked": True}
        # Loads small files with optimized dtypes
        else:
            applicable_dtype = {col: DTYPE_MAP[col] for col in USECOLS_MAP[source_name] if col in DTYPE_MAP}
            logs[source_name] = pd.read_csv(
                full_path,
                usecols=USECOLS_MAP[source_name],
                dtype=applicable_dtype
            )
            
    if missing_files:
        raise FileNotFoundError("Missing required CERT files: " + ", ".join(missing_files))
    
    return logs

In [129]:
logs = load_raw_logs(cert_path)

In [130]:
logs["logon"].head()

,date,user,pc,activity
0,01/02/2010 02:19:18,DNS1758,PC-0414,Logon
1,01/02/2010 02:31:12,DNS1758,PC-0414,Logoff
2,01/02/2010 02:34:02,DNS1758,PC-5313,Logon
3,01/02/2010 02:53:30,DNS1758,PC-5313,Logoff
4,01/02/2010 04:07:31,DNS1758,PC-0012,Logon


In [131]:
logs["file"].head()

,date,user,pc,filename,activity
0,01/02/2010 07:19:41,SDH2394,PC-5849,R:\60WBQE7S.doc,File Open
1,01/02/2010 07:21:30,SDH2394,PC-5849,R:\0VGILDW8.pdf,File Write
2,01/02/2010 07:22:11,SDH2394,PC-5849,R:\60WBQE7S.doc,File Copy
3,01/02/2010 07:24:06,SDH2394,PC-5849,R:\22B5gX4\H8Y96RRE.doc,File Write
4,01/02/2010 07:24:45,SDH2394,PC-5849,R:\SDH2394\7XRCV2N5.pdf,File Copy


In [132]:
logs["device"].head()

,date,user,pc,activity
0,01/02/2010 07:17:18,SDH2394,PC-5849,Connect
1,01/02/2010 07:22:42,JKS2444,PC-6961,Connect
2,01/02/2010 07:31:42,CBA1023,PC-1570,Connect
3,01/02/2010 07:33:28,GNT0221,PC-6427,Connect
4,01/02/2010 07:33:55,JKS2444,PC-6961,Disconnect


In [133]:
logs["email"]

{'path': 'C:\\Users\\loera\\Documents\\Datasets\\CERT_r6.2\\email.csv',
 'chunked': True}

In [134]:
logs["http"]

{'path': 'C:\\Users\\loera\\Documents\\Datasets\\CERT_r6.2\\http.csv',
 'chunked': True}

### Normalizing Data Within Shared Columns:

In [135]:
def normalize_shared_columns(df: pd.DataFrame, remove_cols: list=["id"]) -> pd.DataFrame:
    """
    Normalizes CERT log files across commonly shared columns. Additionally drops columns that are deemed irrelevant.
    
    Args:
        df: The raw CERT dataframe (logon, file, device, or email)
        remove_cols: The columns to drop from the original CERT file
        
    Returns:
        pd.Dataframe: A normalized dataframe with consistent identifiers and fields
    """
    # Standardizing column names
    df = df.copy()
    df.columns = df.columns.str.lower().str.strip()
    
    # Renaming date column
    if "date" in df.columns:
        df.rename(columns={"date": "timestamp"}, inplace=True)
        
    if "timestamp" not in df.columns:
        raise KeyError("Expected a 'date' or 'timestamp' column in DataFrame.")
        
    # Converting timestamp to datetime.
    df["timestamp"] = pd.to_datetime(df["timestamp"], format=TIMESTAMP_FORMAT, errors="coerce")
    
    # Dropping rows with invalid timestamps
    df.dropna(axis=0, subset=["timestamp"], inplace=True)
    
    # Creating a 'day' aggregation key column
    df["day"] = df["timestamp"].dt.floor("D")
    
    # Normalizing identifiers
    df["user"] = df["user"].astype(str).str.lower().str.strip()
    df["pc"] = df["pc"].astype(str).str.lower().str.strip()
    
    # Dropping unusable columns
    remove_cols = [col.lower().strip() for col in remove_cols]
    cols_to_drop = [col for col in remove_cols if col in df.columns]
    df.drop(axis=1, columns=cols_to_drop, inplace=True)
    
    # Sorting rows for consistency
    df.sort_values(by=["user", "pc", "timestamp"], inplace=True)
    df.reset_index(drop=True, inplace=True)
    
    return df

In [136]:
# Normalizing eagerly-loaded log files
for name, df in logs.items():
    if isinstance(df, dict) and df.get("chunked"):
        print(f"{name}.csv will be normalized when loaded per-chunk")
        continue
    logs[name] = normalize_shared_columns(df)

email.csv will be normalized when loaded per-chunk
http.csv will be normalized when loaded per-chunk


In [ ]:
# Displaying normalized logon logs
norm_logons  = logs["logon"]
norm_logons.head()

,timestamp,user,pc,activity,day
0,2010-01-04 07:41:00,aab0162,pc-6599,Logon,2010-01-04
1,2010-01-04 18:46:00,aab0162,pc-6599,Logoff,2010-01-04
2,2010-01-05 07:46:00,aab0162,pc-6599,Logon,2010-01-05
3,2010-01-05 18:40:00,aab0162,pc-6599,Logoff,2010-01-05
4,2010-01-06 07:45:00,aab0162,pc-6599,Logon,2010-01-06


In [ ]:
# Displaying normalized file logs
norm_files = logs["file"]
norm_files.head()

,timestamp,user,pc,filename,activity,day
0,2010-01-13 10:34:04,aab0162,pc-6599,C:\ASMWXYUP.pdf,File Open,2010-01-13
1,2010-01-13 12:22:42,aab0162,pc-6599,C:\ASMWXYUP.pdf,File Open,2010-01-13
2,2010-01-18 15:40:26,aab0162,pc-6599,C:\ASMWXYUP.pdf,File Open,2010-01-18
3,2010-03-08 14:49:23,aab0162,pc-6599,C:\ASMWXYUP.pdf,File Open,2010-03-08
4,2010-05-19 13:10:35,aab0162,pc-6599,C:\ASMWXYUP.pdf,File Open,2010-05-19


In [138]:
# Displaying normalized device logs
norm_devices = logs["device"]
norm_devices.head()

,timestamp,user,pc,activity,day
0,2010-01-04 08:26:37,aac0610,pc-1834,Connect,2010-01-04
1,2010-01-04 10:23:08,aac0610,pc-1834,Disconnect,2010-01-04
2,2010-01-05 14:10:01,aac0610,pc-1834,Connect,2010-01-05
3,2010-01-05 14:20:11,aac0610,pc-1834,Disconnect,2010-01-05
4,2010-01-05 15:04:58,aac0610,pc-1834,Connect,2010-01-05


In [141]:
def validate_normalized_files(logs: dict[str, pd.DataFrame], required_cols: set={"user", "pc", "timestamp", "day"}) -> None:
    """
    A sanity-check for normalized CERT logs before feature extraction is performed.
    Chunked sources (email, http) are skipped as they are validated per-chunk during extraction.
    
    Args:
        logs: A dictionary of the format: {log_name: DataFrame | {"path": str, "chunked": True}}
        required_cols: A set of required columns for each CERT log
        
    Returns:
        None:
    """
    for name, df in logs.items():
        # Case where file is marked as large file
        if isinstance(df, dict) and df.get("chunked"):
            print(f"Skipped {name}.csv (validated per chunk during extraction)")
            continue
        # Raises error if required column is missing
        if not required_cols.issubset(df.columns):
            missing = required_cols.difference(df.columns)
            raise ValueError(f"Normalized {name} log is missing column(s): {sorted(missing)}")
        # Raises error if required column is in invalid format
        if str(df["day"].dtype) != "datetime64[ns]":
            raise TypeError(f"Normalized {name} log has invalid 'day' format: {df["day"].dtype}")
        
    print("All CERT logs were properly normalized.")

In [142]:
validate_normalized_files(logs)

Skipped email.csv (validated per chunk during extraction)
Skipped http.csv (validated per chunk during extraction)
All CERT logs were properly normalized.


### Defining Helper Functions for Large Files:

In [143]:
def extract_domain(url: str) -> str:
    """
    Extracts a lowercased domain from a URL-like value.
    
    Args:
        url: Raw url string
        
    Returns:
        str: The domain found within the URL
    """
    if pd.isna(url):
        return ""
    
    url = str(url).strip().lower()
    if not url:
        return ""
    
    if not url.startswith(("http://", "https://")):
        url = "http://" + url
        
    try:
        return urlparse(url).netloc.lower()
    except Exception:
        return ""

In [144]:
def load_log_in_chunks(filepath: str, usecols: list, dtype: dict, chunksize: int=50_000) -> pd.io.parsers.readers.TextFileReader:
    """
    Returns an iterator consisting of DataFrames for processing large CSV files in fixed-size chunks.
    
    Args:
        filepath: Absolute path to the CSV file
        usecols: The columns to load in from the CSV file
        dtype: Dict of dtype assignments for specific columns
        chunksize: Number of rows per chunk
        
    Returns:
        pd.io.parsers.TextFileReader: An iterable of DataFrames, one per chunk
    """
    applicable_dtype = {col: dtype[col] for col in usecols if col in dtype}
    return pd.read_csv(filepath, usecols=usecols, dtype=applicable_dtype, chunksize=chunksize)

In [145]:
def combine_partial_aggregations(partial_list: list, merge_cols: list) -> pd.DataFrame:
    """
    Concatenates a list of per-chunk aggregated DataFrames and sums all count columns across groups.
    
    Args:
        partial_list: A list containing the chunked groupby-aggregated DataFrames with additive count columns
        merge_cols: The groupby key columns such as (user, pc, day)
        
    Returns:
        pd.DataFrame: Combined aggregation with counts correctly summed across all chunks
    """
    combined = pd.concat(partial_list, ignore_index=True)
    return combined.groupby(merge_cols, as_index=False).sum()

In [146]:
def build_unique_count(identity_frames: list, merge_cols: list, value_col: str, output_col: str) -> pd.DataFrame:
    """
    Computes an exact per-group nunique count from accumulated per-chunk DataFrames.
    
    Args:
        identity_frames: Per-chunk deduplicated DataFrames
        merge_cols: The groupby key columns (user, pc, day)
        value_col: The column whose distinct values are being counted
        output_col: Name for the resulting count column
        
    Returns:
        pd.DataFrame: (merge_cols + output_col) with exact nunique counts
    """
    combined = pd.concat(identity_frames, ignore_index=True).drop_duplicates()
    return (
        combined.groupby(merge_cols)[value_col]
        .nunique()
        .reset_index()
        .rename(columns={value_col: output_col})
    )

### Feature Engineering Normalized Data:

In [147]:
def extract_logon_features(norm_df: pd.DataFrame, work_hours: tuple=(9, 17)) -> pd.DataFrame:
    """
    Extracts daily authentication behavior features from logon events to create an aggregated feature table.
    
    Args:
        norm_df: The normalized logon dataframe
        work_hours: The range describing regular work hours based on a 24-hour format
        
    Returns:
        pd.DataFrame: Aggregated logon behavior features per (user, pc, day)
    """
    df = norm_df.copy()
    
    # Defining off-hour logons
    df["hour"] = df["timestamp"].dt.hour
    df["off_hours"] = (df["hour"] < work_hours[0]) | (df["hour"] > work_hours[1])
    
    # Creating groups based on (user, pc, day)    
    grouped = df.groupby(["user", "pc", "day"])
    
    # Deriving logon-related features
    features = grouped.agg(
        logon_count = ("activity", lambda x: (x=="Logon").sum()),
        logoff_count = ("activity", lambda x: (x=="Logoff").sum()),
        off_hours_logon = ("off_hours", "sum")
    ).reset_index()
    
    return features

In [151]:
agg_logons = extract_logon_features(norm_logons, WORK_HOURS)

In [152]:
agg_logons.head()

,user,pc,day,logon_count,logoff_count,off_hours_logon
0,aab0162,pc-6599,2010-01-04,1,1,2
1,aab0162,pc-6599,2010-01-05,1,1,2
2,aab0162,pc-6599,2010-01-06,1,1,2
3,aab0162,pc-6599,2010-01-07,1,1,2
4,aab0162,pc-6599,2010-01-08,1,1,2


In [153]:
def extract_file_features(norm_df: pd.DataFrame, work_hours: tuple=(9, 17)) -> pd.DataFrame:
    """
    Extracts daily file access behavior features to create an aggregated feature table.
    
    Args:
        norm_df: Normalized file activity dataframe
        work_hours: The range describing regular work hours based on a 24-hour format
        
    Returns:
        pd.DataFrame: Aggregated file behavior features per (user, pc, day)
    """
    df = norm_df.copy()
    
    # Defining off-hour logons
    df["hour"] = df["timestamp"].dt.hour
    df["off_hours"] = (df["hour"] < work_hours[0]) | (df["hour"] > work_hours[1])
    
    # Creating groups based on (user, pc, day)
    grouped = df.groupby(["user", "pc", "day"])
    
    # Deriving file-related features
    features = grouped.agg(
        file_open_count = ("activity", lambda x: (x=="File Open").sum()),
        file_write_count = ("activity", lambda x: (x=="File Write").sum()),
        file_copy_count = ("activity", lambda x: (x=="File Copy").sum()),
        file_delete_count = ("activity", lambda x: (x=="File Delete").sum()),
        unique_files_accessed = ("filename", "nunique"),
        off_hours_files_accessed = ("off_hours", "sum")
    ).reset_index()
    
    return features

In [154]:
agg_files = extract_file_features(norm_files, WORK_HOURS)

In [155]:
agg_files.head()

,user,pc,day,file_open_count,file_write_count,file_copy_count,file_delete_count,unique_files_accessed,off_hours_files_accessed
0,aab0162,pc-6599,2010-01-13,2,0,0,0,1,0
1,aab0162,pc-6599,2010-01-18,1,0,0,0,1,0
2,aab0162,pc-6599,2010-03-08,1,0,0,0,1,0
3,aab0162,pc-6599,2010-05-19,1,0,0,0,1,0
4,aab0162,pc-6599,2010-05-24,1,0,0,0,1,0


In [156]:
def extract_device_features(norm_df: pd.DataFrame, work_hours: tuple=(9, 17)) -> pd.DataFrame:
    """
    Extracts daily removable media (USB) behavior features to create an aggregated feature table.
    
    Args:
        norm_df: Normalized file activity dataframe
        work_hours: The range describing regular work hours based on a 24-hour format
        
    Returns:
        pd.DataFrame: Aggregated removable media behavior features per (user, pc, day)
    """
    df = norm_df.copy()
    
    # Defining off-hour logons
    df["hour"] = df["timestamp"].dt.hour
    df["off_hours"] = (df["hour"] < work_hours[0]) | (df["hour"] > work_hours[1])
    
    # Creating groups based on (user, pc, day)
    grouped = df.groupby(["user", "pc", "day"])
    
    # Deriving media-related features
    features = grouped.agg(
        usb_insert_count = ("activity", lambda x: (x=="Connect").sum()),
        usb_remove_count = ("activity", lambda x: (x=="Disconnect").sum()),
        off_hours_usb_usage = ("off_hours", "sum")
    ).reset_index()
    
    return features

In [157]:
agg_devices = extract_device_features(norm_devices, WORK_HOURS)

In [158]:
agg_devices.head()

,user,pc,day,usb_insert_count,usb_remove_count,off_hours_usb_usage
0,aac0610,pc-1834,2010-01-04,1,1,1
1,aac0610,pc-1834,2010-01-05,2,2,0
2,aac0610,pc-1834,2010-01-06,1,1,1
3,aac0610,pc-1834,2010-01-07,2,2,0
4,aac0610,pc-1834,2010-01-08,1,1,0


In [159]:
def extract_email_features_chunked(filepath: str, work_hours: tuple = (9, 17), chunksize: int=50_000) -> pd.DataFrame:
    """
    Memory-efficient email feature extraction via chunked CSV reading.
    
    Additive counts are accumulated per chunk then combined via `combine_partial_aggregations`.
    Unique recipients are tracked via `build_unique_count`.
    
    Args:
        filepath: Absolute path to email.csv
        work_hours: The range describing regular work hours based on a 24-hour format
        chunksize: Number of rows per chunk
        
    Returns:
        pd.DataFrame: Aggregated email behavior features per (user, pc, day)
    """
    MERGE_COLS = ["user", "pc", "day"]
    partial_aggs = []
    identity_frames = []

    for chunk in load_log_in_chunks(filepath, USECOLS_MAP["email"], DTYPE_MAP, chunksize):
        chunk = normalize_shared_columns(chunk)
        
        # Defining off-hours emails
        chunk["hour"] = chunk["timestamp"].dt.hour
        chunk["off_hours"] = (chunk["hour"] < work_hours[0]) | (chunk["hour"] > work_hours[1])
        
        # External email heuristic
        chunk["external_email"] = ~chunk["to"].str.contains(INTERNAL_EMAIL_DOMAIN, na=False)

        # Deriving email-related features per chunk
        partial = chunk.groupby(MERGE_COLS).agg(
            emails_sent = ("to", "count"),
            external_emails = ("external_email", "sum"),
            attachments_sent= ("attachments", lambda x: x.notnull().sum()),
            off_hours_emails= ("off_hours", "sum"),
        ).reset_index()

        # Accumulating deduplicated tuples (user, pc, day, to) for unique recipient counting
        identity_frames.append(chunk[MERGE_COLS + ["to"]].drop_duplicates())
        partial_aggs.append(partial)
        del chunk, partial

    # Computes aggregation and unique count operations across all chunks
    combined = combine_partial_aggregations(partial_aggs, MERGE_COLS)
    unique_recipients  = build_unique_count(identity_frames, MERGE_COLS, "to", "unique_recipients")

    return combined.merge(unique_recipients, on=MERGE_COLS, how="left")

In [160]:
agg_emails = extract_email_features_chunked(logs["email"]["path"], WORK_HOURS)

In [161]:
agg_emails.head()

,user,pc,day,emails_sent,external_emails,attachments_sent,off_hours_emails,unique_recipients
0,aab0162,pc-6599,2010-01-04,9,1,1,5,5
1,aab0162,pc-6599,2010-01-05,9,1,2,0,8
2,aab0162,pc-6599,2010-01-06,9,0,1,8,8
3,aab0162,pc-6599,2010-01-07,9,2,1,5,7
4,aab0162,pc-6599,2010-01-08,9,2,0,0,7


In [162]:
def extract_http_features_chunked(filepath: str, work_hours: tuple=(9, 17), chunksize: int=50_000) -> pd.DataFrame:
    """
    Memory-efficient HTTP feature extraction via chunked CSV reading.
    
    Additive counts are accumulated per chunk then combined via `combine_partial_aggregations`.
    Unique domains visited are tracked via `build_unique_count`.
    
    Args:
        filepath: Absolute path to http.csv
        work_hours: The range describing regular work hours based on a 24-hour format
        chunksize: Number of rows per chunk
        
    Returns:
        pd.DataFrame: Aggregated web browsing features per (user, pc, day)
    """
    MERGE_COLS = ["user", "pc", "day"]
    partial_aggs = []
    identity_frames = []

    for chunk in load_log_in_chunks(filepath, USECOLS_MAP["http"], DTYPE_MAP, chunksize):
        chunk = normalize_shared_columns(chunk)

        # Defining off-hours web activity
        chunk["hour"] = chunk["timestamp"].dt.hour
        chunk["off_hours"] = (chunk["hour"] < work_hours[0]) | (chunk["hour"] > work_hours[1])

        # Deriving http-related features per chunk
        chunk["url"] = chunk["url"].astype(str).str.strip().str.lower()
        chunk["url_length"] = chunk["url"].str.len()
        chunk["domain"] = chunk["url"].apply(extract_domain)
        
        chunk["is_www_visit"] = (chunk["activity"] == "WWW Visit").astype(int)
        chunk["is_www_download"] = (chunk["activity"] == "WWW Download").astype(int)
        chunk["is_www_upload"] = (chunk["activity"] == "WWW Upload").astype(int)
               
        chunk["is_job_site"] = chunk["domain"].isin(JOB_DOMAINS).astype(int)
        chunk["is_cloud_storage"] = chunk["domain"].isin(CLOUD_STORAGE_DOMAINS).astype(int)
        chunk["is_suspicious_domain"] = chunk["domain"].isin(SUSPICIOUS_DOMAINS).astype(int)
        chunk["is_long_url"] = (chunk["url_length"] >= LONG_URL_THRESHOLD).astype(int)

        # Grouping chunk and aggregating features
        grouped_chunk = chunk.groupby(MERGE_COLS, dropna=False)
        
        agg_chunk = grouped_chunk.agg(
            http_total_requests = ("url", "count"),
            http_visit_count = ("is_www_visit", "sum"),
            http_download_count = ("is_www_download", "sum"),
            http_upload_count = ("is_www_upload", "sum"),
            jobsite_visits = ("is_job_site", "sum"),
            cloud_storage_visits = ("is_cloud_storage", "sum"),
            suspicious_site_visits = ("is_suspicious_domain", "sum"),
            off_hours_http = ("off_hours", "sum"),
            long_url_count = ("is_long_url", "sum"),
        ).reset_index()

        # Accumulating deduplicated tuples (user, pc, day, domain) for unique domain counting
        identity_frames.append(chunk[MERGE_COLS + ["domain"]].drop_duplicates())
        partial_aggs.append(agg_chunk)
        del chunk, agg_chunk

    # Computes aggregation and unique count operations across all chunks
    combined = combine_partial_aggregations(partial_aggs, MERGE_COLS)
    unique_domains = build_unique_count(identity_frames, MERGE_COLS, "domain", "unique_domains_visited")

    return combined.merge(unique_domains, on=MERGE_COLS, how="left")

In [163]:
agg_http = extract_http_features_chunked(logs["http"]["path"], WORK_HOURS)

In [164]:
agg_http.head()

,user,pc,day,http_total_requests,http_visit_count,http_download_count,http_upload_count,jobsite_visits,cloud_storage_visits,suspicious_site_visits,off_hours_http,long_url_count,unique_domains_visited
0,aab0162,pc-6599,2010-01-04,95,95,0,0,0,0,0,32,34,16
1,aab0162,pc-6599,2010-01-05,95,95,0,0,0,0,0,22,27,14
2,aab0162,pc-6599,2010-01-06,95,95,0,0,0,0,0,15,54,19
3,aab0162,pc-6599,2010-01-07,95,95,0,0,0,0,0,25,44,26
4,aab0162,pc-6599,2010-01-08,95,95,0,0,0,0,0,30,42,10


In [165]:
feature_tables = [agg_logons, agg_files, agg_devices, agg_emails, agg_http]

### Merging Aggregated Feature Tables Into One Behavioral Matrix:

In [166]:
def merge_behavioral_features(feature_tables: list[pd.DataFrame], merge_cols: list=["user", "pc", "day"]) -> pd.DataFrame:
    """
    Merges multiple aggregated behavioral feature tables into a single dataset.
    
    Args:
        feature_tables: A list of aggregated features dataframes to merge, where each table must contain the merge columns
        merge_cols: Key columns to merge on
        
    Returns:
        pd.DataFrame: A unified table with missing activity filled with zeros
    """
    # Filtering out empty tables
    valid_tables = [df for df in feature_tables if ((df is not None) and (not df.empty))]
    
    if not valid_tables:
        raise ValueError("No valid feature tables provided for merging")

    merged_df = valid_tables[0].copy()
    
    # Iteratively merging tables
    for df in valid_tables[1:]:
        merged_df = merged_df.merge(df, on=merge_cols, how="outer")
        
    # Identifying feature columns, excluding identifiers
    feature_cols = [col for col in merged_df.columns if col not in merge_cols]
    
    # Filling missing feature values with zero
    merged_df[feature_cols] = merged_df[feature_cols].fillna(0).astype("int32")
    merged_df[feature_cols] = merged_df[feature_cols].astype("int32")
    
    # Sorting dataframe for consistency
    merged_df.sort_values(by=merge_cols, inplace=True)
    merged_df.reset_index(drop=True, inplace=True)
    
    # Ensuring no duplicate rows are 
    if merged_df.duplicated(subset=merge_cols).any():
        raise ValueError("Duplicate rows detected after merging feature tables.")
    
    return merged_df

In [167]:
behavioral_matrix = merge_behavioral_features(feature_tables)

In [168]:
behavioral_matrix.head()

,user,pc,day,logon_count,logoff_count,off_hours_logon,file_open_count,file_write_count,file_copy_count,file_delete_count,...,http_total_requests,http_visit_count,http_download_count,http_upload_count,jobsite_visits,cloud_storage_visits,suspicious_site_visits,off_hours_http,long_url_count,unique_domains_visited
0,aab0162,pc-6599,2010-01-04,1,1,2,0,0,0,0,...,95,95,0,0,0,0,0,32,34,16
1,aab0162,pc-6599,2010-01-05,1,1,2,0,0,0,0,...,95,95,0,0,0,0,0,22,27,14
2,aab0162,pc-6599,2010-01-06,1,1,2,0,0,0,0,...,95,95,0,0,0,0,0,15,54,19
3,aab0162,pc-6599,2010-01-07,1,1,2,0,0,0,0,...,95,95,0,0,0,0,0,25,44,26
4,aab0162,pc-6599,2010-01-08,1,1,2,0,0,0,0,...,95,95,0,0,0,0,0,30,42,10


### Feature Engineering PC-Related Data:

In [169]:
def add_pc_features(df: pd.DataFrame, min_history: int=10) -> pd.DataFrame:
    """
    Adds PC-derived behavioral history to an aggregated behavioral matrix.
    
    Args:
        df: The behavioral matrix
        min_history: Minimum prior observations required before flagging new PC usage as abnormal activity
        
    Returns:
        pd.DataFrame: A behavioral matrix with added user-to-PC behavioral history
    """
    df = df.copy()
    df.sort_values(by=["user", "day", "pc"], inplace=True)
    df.reset_index(drop=True, inplace=True)
    
    # Tracking historical PC usage counts
    df["pc_prior_use_count"] = df.groupby(["user", "pc"]).cumcount()
    df["user_total_prior_days"] = df.groupby("user").cumcount()
    df["pc_seen_before"]  = (df["pc_prior_use_count"] > 0).astype(int)
    
    df["pc_prior_use_ratio"] = np.where(
        df["user_total_prior_days"] > 0,
        df["pc_prior_use_count"] / df["user_total_prior_days"],
        0.0
    )
    
    # Identifying a user's primary PC (i.e., most-used PC)
    primary_pc_map = (
        df.groupby(["user", "pc"])
        .size()
        .reset_index(name="count")
        .sort_values(["user", "count", "pc"], ascending=[True, False, True])
        .drop_duplicates(subset=["user"])
        .set_index("user")["pc"]
        .to_dict()
    )
    
    df["pc_is_primary"] = (df["pc"] == df["user"].map(primary_pc_map)).astype(int)
    
    # Tracking the number of distinct PC's previously used
    df["distinct_pcs_prior"] = (
        df.groupby(["user"])["pc"]
        .transform(lambda s: [len(set(s.iloc[:i])) for i in range(len(s))])
    )
    
    # Tracking the number of unique PC's used on a given day
    same_day_counts = (
        df.groupby(["user", "day"])["pc"]
        .transform("nunique")
    )
    
    df["multi_pc_day"] = same_day_counts
    
    # Identifies new PC usage after an established history
    df["new_pc_after_history"] = (
        (df["pc_prior_use_count"] == 0) &
        (df["user_total_prior_days"] >= min_history)
    ).astype(int)
    
    df.drop(columns=["user_total_prior_days"], inplace=True)
    
    return df

In [170]:
layer_a_ueba_dataset = add_pc_features(behavioral_matrix)

In [172]:
layer_a_ueba_dataset.head()

,user,pc,day,logon_count,logoff_count,off_hours_logon,file_open_count,file_write_count,file_copy_count,file_delete_count,...,off_hours_http,long_url_count,unique_domains_visited,pc_prior_use_count,pc_seen_before,pc_prior_use_ratio,pc_is_primary,distinct_pcs_prior,multi_pc_day,new_pc_after_history
0,aab0162,pc-6599,2010-01-04,1,1,2,0,0,0,0,...,32,34,16,0,0,0.0,1,0,1,0
1,aab0162,pc-6599,2010-01-05,1,1,2,0,0,0,0,...,22,27,14,1,1,1.0,1,1,1,0
2,aab0162,pc-6599,2010-01-06,1,1,2,0,0,0,0,...,15,54,19,2,1,1.0,1,1,1,0
3,aab0162,pc-6599,2010-01-07,1,1,2,0,0,0,0,...,25,44,26,3,1,1.0,1,1,1,0
4,aab0162,pc-6599,2010-01-08,1,1,2,0,0,0,0,...,30,42,10,4,1,1.0,1,1,1,0


### Saving Layer A UEBA Dataset:

In [ ]:
def save_dataset(dataset: pd.DataFrame, filename: str, output_dir: str=DEFAULT_OUTPUT_DIR) -> str:
    """
    Saves the UEBA-enhanced dataset to the specified path as a CSV file.
    
    Args:
        dataset: The UEBA-enhanced dataset
        file_name: The desired name of the CSV dataset
        output_dir: Directory where processed outputs are saved
        
    Returns:
        str: Full path to the saved dataset
    """
    # Ensures directory exists
    save_path = os.path.join(os.getcwd(), output_dir)
    os.makedirs(save_path, exist_ok=True)
    
    # Creates full file path
    file_path = os.path.join(save_path, filename)
    
    # Saving the dataset
    dataset.to_csv(file_path)
    print(f"Dataset successfully saved to: {file_path}")
    
    return file_path

In [ ]:
save_path = save_dataset(layer_a_ueba_dataset, filename="ueba_dataset_3a.csv")

'processed_datasets'

## **Layer B**

### Integrating PC and UEBA-Specific Enhancements:

In [ ]:
def apply_ueba_enhancements(df: pd.DataFrame, id_cols: list=["user", "pc", "day"], rolling_window: int=7) -> pd.DataFrame:
    """
    Applying UEBA-specific enhancements to a behavioral matrix such as:
    - Per-user causal z-score deviations based only on prior history
    - Causal rolling mean deltas that exclude the current row
    - Cross-channel risk flags
    
    Args:
        df: A merged behavioral matrix
        id_cols: Identifier columns to exclude from feature calculation
        rolling_window: Window size in days for rolling statistics
        
    Returns:
        pd.DataFrame: An enhanced UEBA-ready feature matrix
    """
    df = df.copy()
    df.sort_values(by=["user", "day"], inplace=True)
    
    feature_cols = [col for col in df.columns if col not in id_cols]
    
    # Defining per-user causal z-score deviations
    for col in feature_cols:
        grouped = df.groupby("user")[col]
        
        # Building baselines from prior observations
        prior_mean = grouped.transform(lambda x: x.shift(1).expanding(min_periods=1).mean())
        prior_std = grouped.transform(lambda x: x.shift(1).expanding(min_periods=2).std())
        
        # Avoiding division by zero
        prior_std.replace(0, np.nan, inplace=True)
        
        # Computing z-scores
        df[f"{col}_zscore"] = ((df[col] - prior_mean) / prior_std).fillna(0.0)
        
    # Defining causal rolling temporal deltas
    for col in feature_cols:
        prior_rolling_mean = (
            df.groupby("user")[col]
            .transform(lambda x: x.shift(1).rolling(window=rolling_window, min_periods=1).mean())
        )
        
        # First observations have no prior window, so the delta defaults to 0.
        df[f"{col}_rolling_delta"] = (df[col] - prior_rolling_mean).fillna(0.0)
        
    # Defining cross-channel risk flags
    df["usb_file_activity_flag"] = ((df.get("usb_insert_count", 0) > 0) & (df.get("file_write_count", 0) > 0)).astype(int)
    
    df["off_hours_activity_flag"] = ((df.filter(like="off_hours").sum(axis=1) > 0)).astype(int)
    
    df["external_comm_activity_flag"] = (df.get("external_emails", 0) > 0).astype(int)
    
    df["non_primary_pc_usb_flag"] = ((df["pc_is_primary"] == 0) & (df.get("usb_insert_count", 0) > 0)).astype(int)
    
    df["non_primary_pc_file_copy_flag"] = ((df["pc_is_primary"] == 0) & (df.get("file_copy_count", 0) > 0)).astype(int)
    
    df["non_primary_pc_off_hours_flag"] = ((df["pc_is_primary"] == 0) & (df.filter(like="off_hours").sum(axis=1) > 0)).astype(int)

    return df

In [ ]:
ueba_matrix = apply_ueba_enhancements(pc_behavioral_matrix)

In [ ]:
ueba_matrix.head()

### Saving the Final UEBA-Enhanced Dataset:

In [ ]:
save_dataset(ueba_matrix, "ueba_dataset_2.csv")